In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import csv
import math
import statistics
import pickle
from itertools import combinations
from matplotlib import pyplot as plt
import igraph as ig
import math

In [ ]:
connections_df = pd.read_csv("../data_drosophila_adulta/connections.csv")
connections_df = connections_df.drop(['neuropil', 'syn_count', 'nt_type'], axis=1)
connections_df['pre_root_id'] = connections_df['pre_root_id'].astype('str')
connections_df['post_root_id'] = connections_df['post_root_id'].astype('str')
connections_df = connections_df.drop_duplicates()

In [ ]:
coord = pd.read_csv("../emisferi/adulta/csv/Coordinate/Coordinate 2.0.csv", header=0)

In [ ]:
coord['Id'] = coord['Id'].astype('str')

In [ ]:
superclasses = pd.read_csv("../data_drosophila_adulta/classification.csv", header=0)

In [ ]:
superclasses['root_id'] = superclasses['root_id'].astype('str')

In [ ]:
G = ig.Graph.TupleList(connections_df.itertuples(index=False), directed=True)

In [ ]:
for i, vertex in enumerate(G.vs):
    if i%1000 == 0:
        print(i)
    root_id = vertex["name"]  # Supponendo che 'name' contenga l'identificatore del nodo

    # Assegna gli attributi dai dataframe
    vertex["superclass"] = superclasses[superclasses['root_id'] == root_id]['super_class'].iloc[0]
    vertex["hemisphere"] = superclasses[superclasses['root_id'] == root_id]['side'].iloc[0]
    vertex["x"] = coord[coord['Id'] == root_id]['x'].iloc[0]
    vertex["y"] = coord[coord['Id'] == root_id]['y'].iloc[0]
    vertex["z"] = coord[coord['Id'] == root_id]['z'].iloc[0]

In [ ]:
coord_dict = coord.set_index("Id")[["x", "y", "z"]].to_dict(orient="index")

for i, edge in enumerate(G.es):
    if i % 100000 == 0:
        print(i)

    u, v = G.vs[edge.source]["name"], G.vs[edge.target]["name"]
    p1, p2 = coord_dict[u], coord_dict[v]

    distanza = math.sqrt((p2["x"] - p1["x"])**2 + (p2["y"] - p1["y"])**2 + (p2["z"] - p1["z"])**2)
    edge["weight"] = round(distanza, 3)

In [ ]:
for i, edge in enumerate(G.es):
    print(edge.target)
    break

In [ ]:
for i, v in enumerate(G.vs):
    print(v)
    break

In [ ]:
G.vcount()

In [ ]:
G.write_graphml("weightedGraph_adult.graphml")

In [ ]:
G = ig.Graph.Read_GraphML("weightedGraph_adult.graphml")

## Media delle distanze per area

Partendo dalla matrice di adiacenza pesata, calcolare la media delle distanze per area. Questa media va calcolata sommando i pesi degli archi uscenti dai nodi diviso il numero degli stessi archi. Essendo la rete diretta, nel caso in cui fossero presenti due archi tra gli stessi nodi, ad esempio A->B e B->A, consideriamo questa distanza due volte.

In [ ]:
superclasses = list({v["superclass"] if "superclass" in v.attributes() else "unknown" for v in G.vs})

In [ ]:
nodi_rossi = [v.index for v in G.vs if v["superclass"] =='optic']

In [ ]:
edges_superclass = [e.index for e in G.es if e.source in nodi_rossi]

In [ ]:
dist_info_superclasses = {}

for superclass in superclasses:
    print(superclass)
    nodes_superclass = [v["name"] for v in G.vs if v["superclass"] == superclass]
    edges_superclass = [(G.vs[edge.source]["name"], G.vs[edge.target]["name"]) 
                        for edge in G.es if G.vs[edge.source]["name"] in nodes_superclass]
    edges_weight = []

    # Iterare sugli archi che appartengono ai nodi della superclass
    for (s, t) in edges_superclass:
        print((s,t))
        # Recuperare gli indici dei nodi da 's' e 't' (nodi sono stringhe)
        s_index = G.vs.find(name=s).index
        t_index = G.vs.find(name=t).index
        
        # Trova l'indice dell'arco tra s e t
        edge = G.get_eid(s_index, t_index)  # Ottieni l'indice dell'arco tra i nodi
        edge_data = G.es[edge]  # Ottieni i dati dell'arco
        edges_weight.append(edge_data["weight"])  # Aggiungi il peso dell'arco
        

    superclass_info = {
        'edges': edges_superclass,
        'edges_weight': edges_weight,
        'min': min(edges_weight),
        'max': max(edges_weight),
        'average': sum(edges_weight)/len(edges_weight),
        'median': statistics.median(edges_weight)
    }
    dist_info_superclasses[superclass] = superclass_info
        
with open('Adulta/dist_info_superclasses.pickle', 'wb') as f:
    pickle.dump(dist_info_superclasses, f)

A questo punto trovo i potenziali colonist per ogni area in questo modo: quei nodi che hanno almeno un arco uscente la cui distanza è maggiore della distanza media dell'area appena calcolata.

In [ ]:
with open('Adulta/dist_info_superclasses.pickle', 'rb') as f:
    dist_info_superclasses = pickle.load(f)

In [ ]:
# passo 0.05 da 2.4 a 3
average_grid_search = [round(2.4+(x*0.05), 2) for x in range(0, 13)]
average_grid_search

In [ ]:
p_iteration = {}

for p in average_grid_search:
    print(p)
    p_iteration[p] = {}
    for superclass in superclasses:
        info_superclass = dist_info_superclasses[superclass]
        nodes_superclass = [v["name"] for v in G.vs if v["superclass"] == superclass]
    
        candidate_colonists = set()
    
        for source in nodes_superclass:
            source_index = G.vs.find(name=source).index
            
            out_edges = G.es.select(_source=source_index)
            
            for edge in out_edges:
                if edge["weight"] > p * info_superclass['average']:
                    candidate_colonists.add(source)
                    break
        with open('Adulta/candidate_colonists_' + superclass + '_' + str(p) + '.pickle', 'wb') as f:
            pickle.dump(candidate_colonists, f)
        # print(superclass + ': ' + str(len(candidate_colonists)))

        p_iteration[p][superclass] = len(candidate_colonists)
    p_iteration[p]['tot'] = sum(p_iteration[p].values())
    print(p_iteration[p]['tot'])

In [ ]:
intestazioni_righe = list(next(iter(p_iteration.values())).keys())

with open('Adulta/grid_search.csv', mode="w", newline="", encoding="utf-8") as file_csv:
    writer = csv.writer(file_csv)

    writer.writerow([""] + list(p_iteration.keys()))

    for riga in intestazioni_righe:
        writer.writerow([riga] + [p_iteration[colonna][riga] for colonna in p_iteration])


## Numero di target per ogni colonist candidato

In [ ]:
# num_target_nodes = {}
# target_nodes_dict = {}

for p in average_grid_search:
    print(p)
    p_iteration[p] = {}
    target_nodes_dict = {}
    num_target_nodes = {}
    for superclass in superclasses:
        with open('Adulta/dist_info_superclasses.pickle', 'rb') as f:
            dist_info_superclasses = pickle.load(f)
        info_superclass = dist_info_superclasses[superclass]
        
        with open('Adulta/candidate_colonists_' + superclass + '_' + str(p) + '.pickle', 'rb') as f:
            candidate_colonists = pickle.load(f)
            
        for source in candidate_colonists:
            out_edges = list(G.out_edges(source, data=True))
            target_nodes = set()
            for edge in out_edges:
                if edge[2]['weight'] > p*info_superclass['average']:
                    target_nodes.add(edge[1])
            if len(target_nodes)>1:
                target_nodes_dict[source] = list(target_nodes)
            num_target_nodes[source] = len(target_nodes)
        
    with open('Adulta/candidate_colonists_targets' + str(p) + '.pickle', 'wb') as f:
                pickle.dump(target_nodes_dict, f)

In [ ]:
with open('Adulta/num_target_nodes.csv', mode="w", newline="", encoding="utf-8") as file_csv:
    writer = csv.writer(file_csv)

    writer.writerow(["Node", "Num target nodes"])

    for key, value in num_target_nodes.items():
        writer.writerow([key] + [value])

## Distanza media della rete

In [ ]:
tot_dist = list(nx.get_edge_attributes(G, 'weight').values())
avg_dist = sum(tot_dist) / len(tot_dist)

In [ ]:
avg_dist

## Dai colonist potenziali ai colonist effettivi

Prendo un potenziale colonist, si considerano i suoi target e si costruisce il grafo totlmente connesso i cui archi sono pesati dalla distanza. Si rimuovono tutti gli archi il cui meso è maggiore della distanza media dell'intero grafo. Sul grafo così ottenuto si calcolano le componenti connesse. Se c'è almeno una componente connessa di dimensione >= 2, il nodo è colonist. Ogni componente è una colonia e va valutata.

In [ ]:
for p in average_grid_search:
    print(p)
    p_iteration[p] = {}
    verified_colonists = {}

    with open('Adulta/candidate_colonists_targets' + str(p) + '.pickle', 'rb') as f:
            target_nodes_dict = pickle.load(f)

    for candidate, targets in target_nodes_dict.items():
        targets_info = {nodo: G.nodes[nodo] for nodo in targets}
        
        subgraph = nx.Graph()
        subgraph.add_nodes_from(list(targets))
        
        edges = list(combinations(list(targets), 2))
        subgraph.add_edges_from(edges)
        w = {}
        for source, target, data in subgraph.edges(data=True):
            source_x = targets_info[source]['x']
            source_y = targets_info[source]['y']
            source_z = targets_info[source]['z']
            target_x = targets_info[target]['x']
            target_y = targets_info[target]['y']
            target_z = targets_info[target]['z']
            distanza = math.sqrt((source_x - target_x)**2 + (source_y - target_y)**2 + (source_z - target_z)**2)
            data['weight'] = distanza
        to_remove = []
        for source, target, data in subgraph.edges(data=True):
            if data['weight']>avg_dist:
                to_remove.append((source, target))
        
        subgraph.remove_edges_from(to_remove)
    
        prov_connected_components = list(nx.connected_components(subgraph))
        connected_components = []
        
        for connected_component in prov_connected_components:
            if len(connected_component) > 1:
                connected_components.append(connected_component)
    
        if len(connected_components) >= 1:
            verified_colonists[candidate] = connected_components

    with open('Adulta/colonists_' + str(p) + '.pickle', 'wb') as f:
            pickle.dump(verified_colonists, f)

## Grafico scelta parametri

In [ ]:
g = nx.read_graphml("weightedGraph.graphml")

In [ ]:
dict_larva = {}
average_grid_search_larva = [round(2+(x*0.05), 2) for x in range(0, 11)]
for p in average_grid_search_larva:
    with open('Larva/colonists_' + str(p) + '.pickle', 'rb') as f:
            verified_colonists = pickle.load(f)
    #dict_larva[p] = len(verified_colonists)/g.number_of_nodes()*100
    dict_larva[p] = len(verified_colonists)

In [ ]:
dict_adulta = {}
for p in average_grid_search:
    with open('Adulta/colonists_' + str(p) + '.pickle', 'rb') as f:
            verified_colonists = pickle.load(f)
    #dict_adulta[p] = len(verified_colonists)/G.number_of_nodes()*100
    dict_adulta[p] = len(verified_colonists)

In [ ]:
dict_adulta

In [ ]:
dict_larva

In [ ]:
237*100/g.number_of_nodes()

In [ ]:
5654*100/G.number_of_nodes()

In [ ]:
x1, y1 = zip(*sorted(dict_larva.items()))
x2, y2 = zip(*sorted(dict_adulta.items()))

plt.figure(figsize=(8, 6))
plt.plot(x1, y1, label='Larva', marker='o')  # Linea per data1

plt.xlabel('Times the average')
plt.ylabel('Percentage of colonists')
plt.title('')
plt.legend()  
plt.grid(True)

plt.show()

In [ ]:
x1, y1 = zip(*sorted(dict_larva.items()))
x2, y2 = zip(*sorted(dict_adulta.items()))

plt.figure(figsize=(8, 6))
plt.plot(x2, y2, label='Adult', marker='s')  # Linea per data2

plt.xlabel('Times the average')
plt.ylabel('Percentage of colonists')
plt.title('')
plt.legend()  
plt.grid(True)

plt.show()

In [ ]:
len(verified_colonists)

In [ ]:
len(verified_colonists)*100/len(G.nodes)

In [ ]:
nodes = [nodo for nodo, attr in g.nodes(data=True) if attr.get('superclass') =='central']
len(nodes)

In [ ]:
edges = list(G.edges(data=True))

with open('Data/edges_adult.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    # Scrivi l'intestazione delle colonne
    writer.writerow(['source', 'target', 'weight'])
    
    # Scrivi i dati riga per riga
    for source, target, attributes in edges:
        writer.writerow([source, target, attributes['weight']])

In [ ]:
list(G.nodes(data=True))[0]

In [ ]:
nodes = list(G.nodes(data=True))

with open('Data/nodes_adult.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    # Scrivi l'intestazione delle colonne
    writer.writerow(['id', 'celltype', 'additional_annotations', 'level_7_cluster', 'x', 'y', 'z', 'hemisphere'])
    
    # Scrivi i dati riga per riga
    for node, attributes in nodes:
        writer.writerow([node, attributes['celltype'], attributes['additional_annotations'], attributes['level_7_cluster'], 
                         attributes['x'], attributes['y'], attributes['z'], attributes['hemisphere']])